# EnergyPredict

This code downloads real electricity data, cleans it, builds features, analyzes it and makes charts.

**What we're working with:**
- Electricity demand + solar/wind generation for California (from the EIA, a US government energy data agency)
- Weather for the same period (from Open-Meteo)

**Before you start:** get a free EIA API key at https://www.eia.gov/opendata/register.php.

**Pckages to install:** pandas, numpy, requests, matplotlib

## Import the tools we need

In [ ]:
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt

# Paste your free EIA key here
api_key = "PUT_YOUR_EIA_API_KEY_HERE"


## Download electricity demand data

We're downloading one month at a time, because the EIA website only lets us ask for 5000 rows per request.

In [ ]:
demand_rows = []

months = pd.date_range("2025-01-01", "2025-12-31", freq="MS")

for month_start in months:
    month_end = month_start + pd.offsets.MonthEnd(1) + pd.Timedelta(hours=23)

    url = "https://api.eia.gov/v2/electricity/rto/region-data/data/"
    params = {
        "api_key": api_key,
        "frequency": "hourly",
        "data[0]": "value",
        "facets[respondent][]": "CISO",   # CISO = California's grid operator
        "facets[type][]": "D",            # D = demand
        "start": month_start.strftime("%Y-%m-%dT%H"),
        "end": month_end.strftime("%Y-%m-%dT%H"),
        "length": 5000,
    }

    response = requests.get(url, params=params)
    data = response.json()["response"]["data"]
    demand_rows.extend(data)
    print("Downloaded", month_start.strftime("%B"), "-", len(data), "rows")

demand_df = pd.DataFrame(demand_rows)[["period", "value"]]
demand_df.columns = ["timestamp", "demand_MW"]
demand_df["timestamp"] = pd.to_datetime(demand_df["timestamp"], utc=True)
demand_df["demand_MW"] = demand_df["demand_MW"].astype(float)

demand_df.head()


## Download solar and wind generation

Same as demand, but a different `type` of data. The function can be used to download any type of data and we use it twice. Once for solar and once for wind. 

In [ ]:
def download_fuel_type(fuel_code, fuel_name):
    rows = []
    for month_start in months:
        month_end = month_start + pd.offsets.MonthEnd(1) + pd.Timedelta(hours=23)

        url = "https://api.eia.gov/v2/electricity/rto/fuel-type-data/data/"
        params = {
            "api_key": api_key,
            "frequency": "hourly",
            "data[0]": "value",
            "facets[respondent][]": "CISO",
            "facets[fueltype][]": fuel_code,
            "start": month_start.strftime("%Y-%m-%dT%H"),
            "end": month_end.strftime("%Y-%m-%dT%H"),
            "length": 5000,
        }
        response = requests.get(url, params=params)
        data = response.json()["response"]["data"]
        rows.extend(data)

    fuel_df = pd.DataFrame(rows)[["period", "value"]]
    fuel_df.columns = ["timestamp", fuel_name]
    fuel_df["timestamp"] = pd.to_datetime(fuel_df["timestamp"], utc=True)
    fuel_df[fuel_name] = fuel_df[fuel_name].astype(float)
    return fuel_df

solar_df = download_fuel_type("SUN", "solar_MW")
wind_df = download_fuel_type("WND", "wind_MW")

print("Solar rows:", len(solar_df))
print("Wind rows:", len(wind_df))


## Download weather data

Open-Meteo doesn't need an API key. We're using Sacramento, California as a representative weather location.

In [ ]:
weather_url = "https://archive-api.open-meteo.com/v1/archive"
weather_params = {
    "latitude": 38.58,
    "longitude": -121.49,
    "start_date": "2025-01-01",
    "end_date": "2025-12-31",
    "hourly": "temperature_2m,relative_humidity_2m,dew_point_2m,wind_speed_10m",
    "timezone": "UTC",
}

weather_response = requests.get(weather_url, params=weather_params)
weather_json = weather_response.json()["hourly"]

weather_df = pd.DataFrame({
    "timestamp": pd.to_datetime(weather_json["time"], utc=True),
    "temperature_C": weather_json["temperature_2m"],
    "humidity_pct": weather_json["relative_humidity_2m"],
    "dewpoint_C": weather_json["dew_point_2m"],
    "wind_speed_kmh": weather_json["wind_speed_10m"],
})

weather_df.head()


## Combine everything into one table

We now have 4 separate tables (demand, solar, wind, weather) that all share a `timestamp` column. We merge them together so every row has demand + generation + weather for the same hour.

In [ ]:
df = demand_df.merge(solar_df, on="timestamp")
df = df.merge(wind_df, on="timestamp")
df = df.merge(weather_df, on="timestamp")

df = df.sort_values("timestamp").reset_index(drop=True)

print("Shape:", df.shape)
df.head()


## Check the data for problems

Before working with the data, we want to look for missing values and values that don't make sense.

In [ ]:
print("Missing values in each column:")
print(df.isna().sum())

print()
print("Smallest and largest demand values:")
print("Min:", df["demand_MW"].min(), "MW")
print("Max:", df["demand_MW"].max(), "MW")


## Feature creation

The project uses 9 lag columns, 7 rolling-window columns, 9 calendar columns, 6 "cyclical" columns, and 2 targets.

### a. Lag features — demand at earlier points in time

A "lag" just means "the value some number of hours ago." Short lags (1-3h) capture momentum; longer lags (24h, 168h) capture the daily and weekly rhythm.

In [ ]:
lag_hours = [1, 2, 3, 6, 12, 24, 48, 72, 168]   # 168 hours = 1 week

for hours_back in lag_hours:
    column_name = "lag_" + str(hours_back) + "h"
    df[column_name] = df["demand_MW"].shift(hours_back)

df.filter(like="lag_").head()


### b. Rolling-window features

A rolling average smooths out noise; rolling std/max/min tell us how *volatile* demand has been recently, not just its level. We always shift by 1 first.

In [ ]:
shifted_demand = df["demand_MW"].shift(1)   

df["rolling_mean_3h"] = shifted_demand.rolling(3).mean()
df["rolling_mean_6h"] = shifted_demand.rolling(6).mean()
df["rolling_mean_24h"] = shifted_demand.rolling(24).mean()
df["rolling_mean_168h"] = shifted_demand.rolling(168).mean()
df["rolling_std_24h"] = shifted_demand.rolling(24).std()
df["rolling_max_24h"] = shifted_demand.rolling(24).max()
df["rolling_min_24h"] = shifted_demand.rolling(24).min()

df.filter(like="rolling_").head()


### c. Calendar features — extract features like hour, day of week etc from timestamp column

In [ ]:
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek     # 0 = Monday, 6 = Sunday
df["day_of_month"] = df["timestamp"].dt.day
df["day_of_year"] = df["timestamp"].dt.dayofyear
df["week_of_year"] = df["timestamp"].dt.isocalendar().week
df["month"] = df["timestamp"].dt.month
df["quarter"] = df["timestamp"].dt.quarter
df["is_weekend"] = df["day_of_week"] >= 5

# US federal holidays in 2025 - demand behaves differently on these days
holidays_2025 = pd.to_datetime([
    "2025-01-01", "2025-01-20", "2025-02-17", "2025-05-26", "2025-06-19",
    "2025-07-04", "2025-09-01", "2025-10-13", "2025-11-11", "2025-11-27", "2025-12-25",
]).date

df["is_holiday"] = df["timestamp"].dt.date.isin(holidays_2025)

df[["hour", "day_of_week", "day_of_month", "day_of_year", "week_of_year", "month", "quarter", "is_weekend", "is_holiday"]].head()


### d. Cyclical features

Hour 23 (11pm) and hour 0 (midnight) are only 1 hour apart in real life, but as plain numbers, 23 and 0 look *far* apart. Also on a clock, 11pm and midnight are right next to each other. Sine/cosine encoding fixes this by wrapping each cycle (24 hours, 7 days, 12 months) around a circle, so "just before midnight" and "just after midnight" end up close together mathematically too.


In [ ]:
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

df.filter(like="_sin").join(df.filter(like="_cos")).head()


### e. Combine renewables, and set the prediction targets

The project predicts demand two different distances into the future: 24 hours ahead and 48 hours ahead.

In [ ]:
df["renewable_MW"] = df["solar_MW"] + df["wind_MW"]

df["target_t_plus_24h"] = df["demand_MW"].shift(-24)
df["target_t_plus_48h"] = df["demand_MW"].shift(-48)

# Drop rows that don't have enough history/future data to compute the columns above
df = df.dropna().reset_index(drop=True)

print("Shape after ALL feature engineering:", df.shape)
print("Total columns:", len(df.columns))
df.head()


## Analyze the data

In [ ]:
# Basic statistics for the main numeric columns
df[["demand_MW", "temperature_C", "humidity_pct", "solar_MW", "wind_MW"]].describe()


In [ ]:
# Which columns are most correlated with demand?
correlation_with_demand = df[[
    "demand_MW", "temperature_C", "humidity_pct", "solar_MW", "wind_MW", "renewable_MW",
    "hour", "is_weekend", "is_holiday", "lag_1h", "lag_24h", "lag_168h", "rolling_mean_24h",
]].corr()["demand_MW"].sort_values(ascending=False)

correlation_with_demand


In [ ]:
# Weekday vs. weekend demand
df.groupby("is_weekend")["demand_MW"].agg(["mean", "std"])


In [ ]:
# Holiday vs. non-holiday demand
df.groupby("is_holiday")["demand_MW"].agg(["mean", "std"])


In [ ]:
# How "peaky" is each hour vs the daily average?
for lag_hours_to_check in [1, 24, 48, 168]:
    column_name = "lag_" + str(lag_hours_to_check) + "h"
    correlation = df["demand_MW"].corr(df[column_name])
    print("Correlation with demand", lag_hours_to_check, "hours ago:", round(correlation, 3))


In [ ]:
# Highest and lowest demand hours in the whole year
print("Top 5 highest-demand hours:")
print(df.nlargest(5, "demand_MW")[["timestamp", "demand_MW", "temperature_C"]])

print()
print("Top 5 lowest-demand hours:")
print(df.nsmallest(5, "demand_MW")[["timestamp", "demand_MW", "temperature_C"]])


## Visualize 

In [ ]:
# 1. Demand over the whole year
plt.figure(figsize=(12, 4))
plt.plot(df["timestamp"], df["demand_MW"])
plt.title("Electricity Demand Over the Year")
plt.xlabel("Date")
plt.ylabel("Demand (MW)")
plt.show()


In [ ]:
# 2. Average demand by hour of day
hourly_avg = df.groupby("hour")["demand_MW"].mean()

plt.figure(figsize=(8, 4))
plt.plot(hourly_avg.index, hourly_avg.values, marker="o")
plt.title("Average Demand by Hour of Day")
plt.xlabel("Hour (0-23)")
plt.ylabel("Average Demand (MW)")
plt.show()


In [ ]:
# 3. Demand vs. temperature
plt.figure(figsize=(6, 5))
plt.scatter(df["temperature_C"], df["demand_MW"], alpha=0.3)
plt.title("Demand vs. Temperature")
plt.xlabel("Temperature (C)")
plt.ylabel("Demand (MW)")
plt.show()


In [ ]:
# 4. Average demand by month, with variability (std) as error bars
monthly_stats = df.groupby("month")["demand_MW"].agg(["mean", "std"])

plt.figure(figsize=(8, 4))
plt.bar(monthly_stats.index, monthly_stats["mean"], yerr=monthly_stats["std"], capsize=4, color="skyblue")
plt.title("Average Demand by Month (with variability)")
plt.xlabel("Month")
plt.ylabel("Average Demand (MW)")
plt.show()


In [ ]:
# 5. Weekday vs. weekend comparison
weekday_weekend_stats = df.groupby("is_weekend")["demand_MW"].mean()

plt.figure(figsize=(5, 4))
plt.bar(["Weekday", "Weekend"], weekday_weekend_stats.values, color=["steelblue", "indianred"])
plt.title("Weekday vs. Weekend Demand")
plt.ylabel("Average Demand (MW)")
plt.show()


In [ ]:
# 6. Holiday effect
holiday_stats = df.groupby("is_holiday")["demand_MW"].mean()

plt.figure(figsize=(5, 4))
plt.bar(["Non-holiday", "Holiday"], holiday_stats.values, color=["steelblue", "orange"])
plt.title("Holiday vs. Non-holiday Demand")
plt.ylabel("Average Demand (MW)")
plt.show()


In [ ]:
# 7. Solar vs. wind output distribution
plt.figure(figsize=(7, 4))
plt.hist(df["solar_MW"], bins=40, alpha=0.6, label="Solar")
plt.hist(df["wind_MW"], bins=40, alpha=0.6, label="Wind")
plt.title("Solar vs. Wind Output Distribution")
plt.xlabel("MW")
plt.legend()
plt.show()


In [ ]:
# 8. Correlation ranking as a bar chart
correlation_with_demand.drop("demand_MW").sort_values().plot(kind="barh", figsize=(7, 6), color="teal")
plt.title("What Correlates Most with Demand?")
plt.xlabel("Correlation")
plt.show()


In [ ]:
# 9. Extreme events highlighted on the yearly timeline
top5 = df.nlargest(5, "demand_MW")
bottom5 = df.nsmallest(5, "demand_MW")

plt.figure(figsize=(12, 4))
plt.plot(df["timestamp"], df["demand_MW"], linewidth=0.5, color="lightgray")
plt.scatter(top5["timestamp"], top5["demand_MW"], color="red", label="Highest 5", zorder=3)
plt.scatter(bottom5["timestamp"], bottom5["demand_MW"], color="green", label="Lowest 5", zorder=3)
plt.title("Extreme Demand Events")
plt.legend()
plt.show()


## Save the finished dataset

In [ ]:
df.to_csv("energy_data_final.csv", index=False)
print("Saved! Final shape:", df.shape)
print("Total columns:", len(df.columns))
